# Faithfulness Evaluation of Neural Summarization Models
## A Comparative Study of BART, PEGASUS, and Flan-T5

**Course:** Natural Language Processing
**Task:** Automatic Summarization  
**Models:** BART, PEGASUS, Flan-T5  
**Datasets:** CNN/DailyMail, XSum  
**Metrics:** ROUGE, SummaC, Statistical Significance (Wilcoxon)  

**Research Question:**  
> Do different neural summarization models differ significantly in their faithfulness to the source text,
> and does this vary across domains and document lengths?

## Step 1: Install Dependencies

In [ ]:
#Restart session after this cell and continue from cell 2
!pip install -q "datasets==2.14.0" "huggingface_hub==0.16.4" "transformers==4.35.0" "fsspec==2023.6.0" rouge-score sentencepiece
!pip install -q summac scipy

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/NLP_HWp_Results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Results will be saved to: {SAVE_DIR}')

## Step 3: Imports

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from rouge_score import rouge_scorer
from scipy import stats
import pandas as pd
import numpy as np
import json
import torch

device = 0 if torch.cuda.is_available() else -1
print(f'Using device: {"GPU" if device == 0 else "CPU"}')

## Step 4: Load Datasets
We use 100 examples from each dataset.  
All metrics (ROUGE and SummaC) are computed on the same 100 examples for consistency.

In [ ]:
# CNN/DailyMail
print('Loading CNN/DailyMail...')
cnn_dm = load_dataset('abisee/cnn_dailymail', '3.0.0', split='test')
cnn_dm = cnn_dm.select(range(100))
cnn_articles   = list(cnn_dm['article'])
cnn_references = list(cnn_dm['highlights'])
print(f'CNN/DailyMail loaded: {len(cnn_articles)} examples')

# XSum
print('Loading XSum...')
xsum = load_dataset('EdinburghNLP/xsum', split='test')
xsum = xsum.select(range(100))
xsum_articles   = list(xsum['document'])
xsum_references = list(xsum['summary'])
print(f'XSum loaded: {len(xsum_articles)} examples')

# Document length stats
cnn_lengths  = [len(a.split()) for a in cnn_articles]
xsum_lengths = [len(a.split()) for a in xsum_articles]
print(f'\nCNN/DM  — avg length: {np.mean(cnn_lengths):.0f} words, median: {np.median(cnn_lengths):.0f}')
print(f'XSum    — avg length: {np.mean(xsum_lengths):.0f} words, median: {np.median(xsum_lengths):.0f}')

## Step 5: Split by Document Length
We split each dataset into SHORT and LONG documents using the median as threshold.  
This directly addresses the research question about length vs. faithfulness.

In [ ]:
def split_by_length(articles, references, threshold=None):
    """
    Split articles into short and long groups based on word count.
    Uses median as threshold if not specified.
    Returns indices for each group.
    """
    lengths = [len(a.split()) for a in articles]
    if threshold is None:
        threshold = int(np.median(lengths))
    short_idx = [i for i, l in enumerate(lengths) if l < threshold]
    long_idx  = [i for i, l in enumerate(lengths) if l >= threshold]
    return short_idx, long_idx, threshold

cnn_short_idx, cnn_long_idx, cnn_thresh = split_by_length(cnn_articles, cnn_references)
xsum_short_idx, xsum_long_idx, xsum_thresh = split_by_length(xsum_articles, xsum_references)

print(f'CNN/DM  — threshold: {cnn_thresh} words | short: {len(cnn_short_idx)} | long: {len(cnn_long_idx)}')
print(f'XSum    — threshold: {xsum_thresh} words | short: {len(xsum_short_idx)} | long: {len(xsum_long_idx)}')

## Step 6: Summarization Functions

In [ ]:
def generate_summaries(model_name, articles, max_new_tokens=128,
                       min_new_tokens=20, batch_size=4):
    print(f'\nLoading model: {model_name}')
    summarizer = pipeline(
        'summarization',
        model=model_name,
        device=device,
        truncation=True
    )
    summaries = []
    for i in range(0, len(articles), batch_size):
        batch = [a[:2000] for a in articles[i:i+batch_size]]
        try:
            results = summarizer(batch, max_new_tokens=max_new_tokens,
                                 min_new_tokens=min_new_tokens, truncation=True)
            summaries.extend([r['summary_text'] for r in results])
        except Exception as e:
            print(f'Error on batch {i}: {e}')
            summaries.extend(['' for _ in batch])
        if (i // batch_size) % 5 == 0:
            print(f'  Progress: {min(i+batch_size, len(articles))}/{len(articles)}')
    del summarizer
    torch.cuda.empty_cache()
    return summaries


def generate_flant5_summaries(articles, batch_size=4):
    model_name = 'google/flan-t5-base'
    print(f'\nLoading model: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if device == 0:
        model = model.cuda()
    summaries = []
    for i in range(0, len(articles), batch_size):
        batch = articles[i:i+batch_size]
        prompts = [f'Summarize the following article in 2-3 sentences:\n\n{a[:1500]}' for a in batch]
        inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                           truncation=True, max_length=512)
        if device == 0:
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128, min_new_tokens=20)
        summaries.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        if (i // batch_size) % 5 == 0:
            print(f'  Progress: {min(i+batch_size, len(articles))}/{len(articles)}')
    del model
    torch.cuda.empty_cache()
    return summaries

## Step 7: Generate Summaries — All Three Models

In [ ]:
# BART
bart_cnn  = generate_summaries('facebook/bart-large-cnn', cnn_articles)
bart_xsum = generate_summaries('facebook/bart-large-cnn', xsum_articles)

In [ ]:
# PEGASUS
pegasus_cnn  = generate_summaries('google/pegasus-xsum', cnn_articles)
pegasus_xsum = generate_summaries('google/pegasus-xsum', xsum_articles)

In [ ]:
# Flan-T5
flant5_cnn  = generate_flant5_summaries(cnn_articles)
flant5_xsum = generate_flant5_summaries(xsum_articles)

## Step 8: ROUGE Evaluation

In [ ]:
def compute_rouge_scores(predictions, references):
    """Returns per-example ROUGE scores (list of dicts)."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    results = []
    for pred, ref in zip(predictions, references):
        if pred.strip() == '':
            results.append({'rouge1': 0, 'rouge2': 0, 'rougeL': 0})
            continue
        s = scorer.score(ref, pred)
        results.append({
            'rouge1': s['rouge1'].fmeasure,
            'rouge2': s['rouge2'].fmeasure,
            'rougeL': s['rougeL'].fmeasure
        })
    return results

def avg_rouge(scores):
    return {
        'ROUGE-1': round(np.mean([s['rouge1'] for s in scores]), 4),
        'ROUGE-2': round(np.mean([s['rouge2'] for s in scores]), 4),
        'ROUGE-L': round(np.mean([s['rougeL'] for s in scores]), 4)
    }

# Compute per-example scores
bart_cnn_rouge    = compute_rouge_scores(bart_cnn,    cnn_references)
bart_xsum_rouge   = compute_rouge_scores(bart_xsum,   xsum_references)
pegasus_cnn_rouge = compute_rouge_scores(pegasus_cnn, cnn_references)
pegasus_xsum_rouge= compute_rouge_scores(pegasus_xsum,xsum_references)
flant5_cnn_rouge  = compute_rouge_scores(flant5_cnn,  cnn_references)
flant5_xsum_rouge = compute_rouge_scores(flant5_xsum, xsum_references)

rouge_results = {
    'BART (CNN/DM)':    avg_rouge(bart_cnn_rouge),
    'BART (XSum)':      avg_rouge(bart_xsum_rouge),
    'PEGASUS (CNN/DM)': avg_rouge(pegasus_cnn_rouge),
    'PEGASUS (XSum)':   avg_rouge(pegasus_xsum_rouge),
    'Flan-T5 (CNN/DM)': avg_rouge(flant5_cnn_rouge),
    'Flan-T5 (XSum)':   avg_rouge(flant5_xsum_rouge),
}

rouge_df = pd.DataFrame(rouge_results).T
print('=== ROUGE Results ===')
print(rouge_df.to_string())

## Step 9: SummaC Faithfulness Evaluation
All 100 examples evaluated — consistent with ROUGE evaluation.

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

from summac.model_summac import SummaCZS

print('Loading SummaC model...')
summac_model = SummaCZS(
    granularity='sentence',
    model_name='vitc',
    device='cuda' if device == 0 else 'cpu'
)

def compute_summac_scores(articles, summaries):
    """Returns per-example SummaC scores."""
    scores_out = []
    # Process in batches of 10 to avoid memory issues
    for i in range(0, len(articles), 10):
        batch_art = articles[i:i+10]
        batch_sum = summaries[i:i+10]
        valid = [(a, s) for a, s in zip(batch_art, batch_sum) if s.strip() != '']
        if not valid:
            scores_out.extend([0.0] * len(batch_art))
            continue
        arts, sums = zip(*valid)
        batch_scores = summac_model.score(list(arts), list(sums))['scores']
        scores_out.extend(batch_scores)
        if i % 30 == 0:
            print(f'  SummaC progress: {min(i+10, len(articles))}/{len(articles)}')
    return scores_out

print('\nComputing SummaC for BART (CNN/DM)...')
bart_cnn_summac    = compute_summac_scores(cnn_articles,  bart_cnn)
print('Computing SummaC for BART (XSum)...')
bart_xsum_summac   = compute_summac_scores(xsum_articles, bart_xsum)
print('Computing SummaC for PEGASUS (CNN/DM)...')
pegasus_cnn_summac = compute_summac_scores(cnn_articles,  pegasus_cnn)
print('Computing SummaC for PEGASUS (XSum)...')
pegasus_xsum_summac= compute_summac_scores(xsum_articles, pegasus_xsum)
print('Computing SummaC for Flan-T5 (CNN/DM)...')
flant5_cnn_summac  = compute_summac_scores(cnn_articles,  flant5_cnn)
print('Computing SummaC for Flan-T5 (XSum)...')
flant5_xsum_summac = compute_summac_scores(xsum_articles, flant5_xsum)

summac_results = {
    'BART (CNN/DM)':    round(np.mean(bart_cnn_summac), 4),
    'BART (XSum)':      round(np.mean(bart_xsum_summac), 4),
    'PEGASUS (CNN/DM)': round(np.mean(pegasus_cnn_summac), 4),
    'PEGASUS (XSum)':   round(np.mean(pegasus_xsum_summac), 4),
    'Flan-T5 (CNN/DM)': round(np.mean(flant5_cnn_summac), 4),
    'Flan-T5 (XSum)':   round(np.mean(flant5_xsum_summac), 4),
}

summac_df = pd.DataFrame.from_dict(summac_results, orient='index', columns=['SummaC'])
print('\n=== SummaC Results (100 examples each) ===')
print(summac_df.to_string())

## Step 10: Document Length Analysis
We compare model faithfulness on SHORT vs LONG documents.  
This directly answers: *'does faithfulness vary with document length?'*

In [ ]:
def subset_scores(scores, indices):
    return [scores[i] for i in indices]

length_results = {}

for model_name, cnn_summac, xsum_summac in [
    ('BART',     bart_cnn_summac,    bart_xsum_summac),
    ('PEGASUS',  pegasus_cnn_summac, pegasus_xsum_summac),
    ('Flan-T5',  flant5_cnn_summac,  flant5_xsum_summac),
]:
    # CNN/DM
    cnn_short = subset_scores(cnn_summac, cnn_short_idx)
    cnn_long  = subset_scores(cnn_summac, cnn_long_idx)
    # XSum
    xsum_short = subset_scores(xsum_summac, xsum_short_idx)
    xsum_long  = subset_scores(xsum_summac, xsum_long_idx)

    length_results[f'{model_name} CNN/DM Short']  = round(np.mean(cnn_short), 4)
    length_results[f'{model_name} CNN/DM Long']   = round(np.mean(cnn_long), 4)
    length_results[f'{model_name} XSum Short']    = round(np.mean(xsum_short), 4)
    length_results[f'{model_name} XSum Long']     = round(np.mean(xsum_long), 4)

# Format as a readable table
rows = []
for model_name, cnn_summac, xsum_summac in [
    ('BART',     bart_cnn_summac,    bart_xsum_summac),
    ('PEGASUS',  pegasus_cnn_summac, pegasus_xsum_summac),
    ('Flan-T5',  flant5_cnn_summac,  flant5_xsum_summac),
]:
    rows.append({
        'Model': model_name,
        'CNN/DM Short': round(np.mean(subset_scores(cnn_summac, cnn_short_idx)), 4),
        'CNN/DM Long':  round(np.mean(subset_scores(cnn_summac, cnn_long_idx)), 4),
        'XSum Short':   round(np.mean(subset_scores(xsum_summac, xsum_short_idx)), 4),
        'XSum Long':    round(np.mean(subset_scores(xsum_summac, xsum_long_idx)), 4),
    })

length_df = pd.DataFrame(rows).set_index('Model')
print(f'=== SummaC by Document Length ===')
print(f'CNN/DM threshold: {cnn_thresh} words | XSum threshold: {xsum_thresh} words')
print(length_df.to_string())

## Step 11: Statistical Significance Tests
We use the Wilcoxon signed-rank test (non-parametric) to check if differences  
between models are statistically significant (p < 0.05).

In [ ]:
def wilcoxon_test(scores_a, scores_b, name_a, name_b, dataset):
    """Wilcoxon signed-rank test between two models."""
    # Need same length and at least some differences
    n = min(len(scores_a), len(scores_b))
    a, b = scores_a[:n], scores_b[:n]
    if all(x == y for x, y in zip(a, b)):
        return {'Comparison': f'{name_a} vs {name_b}', 'Dataset': dataset,
                'p-value': 1.0, 'Significant': 'No'}
    try:
        stat, p = stats.wilcoxon(a, b)
    except Exception:
        p = 1.0
    return {
        'Comparison': f'{name_a} vs {name_b}',
        'Dataset': dataset,
        'p-value': round(p, 4),
        'Significant': 'Yes (p<0.05)' if p < 0.05 else 'No'
    }

stat_tests = [
    # CNN/DM comparisons
    wilcoxon_test(bart_cnn_summac,    pegasus_cnn_summac,  'BART', 'PEGASUS',  'CNN/DM'),
    wilcoxon_test(bart_cnn_summac,    flant5_cnn_summac,   'BART', 'Flan-T5',  'CNN/DM'),
    wilcoxon_test(pegasus_cnn_summac, flant5_cnn_summac,   'PEGASUS', 'Flan-T5','CNN/DM'),
    # XSum comparisons
    wilcoxon_test(bart_xsum_summac,   pegasus_xsum_summac, 'BART', 'PEGASUS',  'XSum'),
    wilcoxon_test(bart_xsum_summac,   flant5_xsum_summac,  'BART', 'Flan-T5',  'XSum'),
    wilcoxon_test(pegasus_xsum_summac,flant5_xsum_summac,  'PEGASUS', 'Flan-T5','XSum'),
]

stat_df = pd.DataFrame(stat_tests)
print('=== Statistical Significance (Wilcoxon Signed-Rank Test) ===')
print(stat_df.to_string(index=False))

## Step 12: Final Combined Results Table

In [ ]:
final_df = rouge_df.copy()
final_df['SummaC'] = summac_df['SummaC']

print('=== FINAL RESULTS TABLE (100 examples, all metrics) ===')
print(final_df.to_string())

# Save all results
final_df.to_csv(f'{SAVE_DIR}/results_main.csv')
length_df.to_csv(f'{SAVE_DIR}/results_by_length.csv')
stat_df.to_csv(f'{SAVE_DIR}/results_statistical.csv', index=False)
print(f'\nAll results saved to {SAVE_DIR}')

## Step 13: Qualitative Examples


In [ ]:
print('=== QUALITATIVE EXAMPLES (XSum) ===\n')
for i in range(3):
    print(f'--- Example {i+1} ---')
    print(f'Article (first 300 chars):\n{xsum_articles[i][:300]}...')
    print(f'\nReference:\n{xsum_references[i]}')
    print(f'\nBART (SummaC: {round(bart_xsum_summac[i],3)}):\n{bart_xsum[i]}')
    print(f'\nPEGASUS (SummaC: {round(pegasus_xsum_summac[i],3)}):\n{pegasus_xsum[i]}')
    print(f'\nFlan-T5 (SummaC: {round(flant5_xsum_summac[i],3)}):\n{flant5_xsum[i]}')
    print('\n' + '='*60 + '\n')

# Save examples with SummaC scores
examples = []
for i in range(10):
    examples.append({
        'article': xsum_articles[i][:500],
        'reference': xsum_references[i],
        'length_words': len(xsum_articles[i].split()),
        'bart':    {'summary': bart_xsum[i],    'summac': round(bart_xsum_summac[i], 3)},
        'pegasus': {'summary': pegasus_xsum[i], 'summac': round(pegasus_xsum_summac[i], 3)},
        'flant5':  {'summary': flant5_xsum[i],  'summac': round(flant5_xsum_summac[i], 3)},
    })

with open(f'{SAVE_DIR}/qualitative_examples.json', 'w') as f:
    json.dump(examples, f, indent=2)
print(f'Examples saved.')

In [ ]:
## Step 14: Visualization

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Data ──────────────────────────────────────────────────────────────────────
models   = ['BART', 'PEGASUS', 'Flan-T5']

rouge1 = {
    'CNN/DailyMail': [0.3424, 0.2744, 0.2497],
    'XSum':          [0.2103, 0.4629, 0.3400],
}
summac = {
    'CNN/DailyMail': [0.2508, -0.2239, -0.1964],
    'XSum':          [0.4392, -0.3049, -0.3655],
}

# ── Colors ────────────────────────────────────────────────────────────────────
colors_cnn  = ['#2E4057', '#4A7FA5', '#7EB8D4']
colors_xsum = ['#E07B39', '#E8A87C', '#F2C9A8']

x     = np.arange(len(models))
width = 0.18

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'ROUGE-1 vs SummaC Faithfulness: BART, PEGASUS, Flan-T5\n'
    'CNN/DailyMail and XSum — 100 test examples each',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, metric_data, metric_name, ylabel in [
    (axes[0], rouge1, 'ROUGE-1',               'ROUGE-1 F1'),
    (axes[1], summac, 'SummaC Faithfulness',   'SummaC Score'),
]:
    bars_cnn  = ax.bar(x - width/2, metric_data['CNN/DailyMail'],
                       width, label='CNN/DailyMail',
                       color=colors_cnn, edgecolor='white', linewidth=0.5)
    bars_xsum = ax.bar(x + width/2, metric_data['XSum'],
                       width, label='XSum',
                       color=colors_xsum, edgecolor='white', linewidth=0.5)

    for bar in list(bars_cnn) + list(bars_xsum):
        h = bar.get_height()
        va = 'bottom' if h >= 0 else 'top'
        offset = 0.008 if h >= 0 else -0.008
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + offset, f'{h:.3f}',
                ha='center', va=va, fontsize=7.5, color='#333333')

    if metric_name == 'SummaC Faithfulness':
        ax.axhline(0, color='#555555', linewidth=0.8, linestyle='--', alpha=0.7)
        ax.set_ylim(-0.55, 0.65)

    ax.set_title(metric_name, fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.25, linestyle='--')

    patch_cnn  = mpatches.Patch(color=colors_cnn[1],  label='CNN/DailyMail')
    patch_xsum = mpatches.Patch(color=colors_xsum[1], label='XSum')
    ax.legend(handles=[patch_cnn, patch_xsum], fontsize=9,
              loc='upper right', framealpha=0.85)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/figure1_rouge_vs_summac.png', dpi=180, bbox_inches='tight')
plt.show()
print('Figure saved to Google Drive.')